# Class Balance and Data Augmentation
This notebook performs experiments to balance 4 classes and use data augmentation strategy to improve model generalization.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import random
from sklearn.model_selection import train_test_split
from google.colab import drive



drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/CA Notebook/image"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")

Mounted at /content/drive


random seed

In [ ]:
import copy

def seed_everything(seed=42):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

# Early stopping mechanism: Monitor validation loss to prevent model overfitting
class EarlyStopping:
    def __init__(self, patience=7, delta=0):
        self.patience = patience  # Number of epochs to tolerate no improvement
        self.delta = delta        # Minimum threshold for loss improvement
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42)

dataset

In [ ]:
from torch.utils.data import Dataset
class FruitDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        '''
        Custom dataset for fruit classification.
        Image label is inferred from filename prefix.
        '''
        self.image_files = [f for f in os.listdir(root_dir) if f.lower().endswith(".jpg")]
        self.class_map = {"apple": 0, "banana": 1, "orange": 2, "mixed": 3}
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        filename = self.image_files[idx]
        image_path = os.path.join(self.root_dir, filename)
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # Label parsing logic
        label = -1
        for cls, idx_val in self.class_map.items():
            if filename.lower().startswith(cls):
                label = idx_val
                break
        return image, label

In [ ]:
def get_loaders(root_dir, img_size, batch_size):
    transform = transforms.Compose([transforms.Resize((img_size, img_size)), transforms.ToTensor()])
    full_dataset = FruitDataset(root_dir, transform=transform)
    indices = list(range(len(full_dataset)))
    labels = [full_dataset[i][1] for i in indices]
    train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=labels, random_state=42)
    train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(full_dataset, val_idx), batch_size=batch_size, shuffle=False)
    return train_loader, val_loader

cnn

In [ ]:
class FinalFruitCNN(nn.Module):
    def __init__(self, num_classes=4, dropout_rate=0.3):
        super(FinalFruitCNN, self).__init__()
        self.conv_section = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2), # 64x64
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2), # 32x32
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, 2)  # 16x16
        )
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc = nn.Sequential(
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_section(x); x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1); return self.fc(x)

train and evaluate

In [ ]:
def train_one_epoch_full(model, loader, criterion, optimizer, device):
    model.train(); total_loss, correct, total = 0, 0, 0
    for imgs, lbs in loader:
        imgs, lbs = imgs.to(device), lbs.to(device)
        outputs = model(imgs); loss = criterion(outputs, lbs)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == lbs).sum().item(); total += lbs.size(0)
    return correct / total, total_loss / len(loader)

def evaluate_full(model, loader, criterion, device):
    model.eval(); total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, lbs in loader:
            imgs, lbs = imgs.to(device), lbs.to(device)
            outputs = model(imgs); loss = criterion(outputs, lbs)
            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == lbs).sum().item(); total += lbs.size(0)
    return correct / total, total_loss / len(loader)

128*128 + 3 layer + dropout 0.3

In [ ]:
# ==========================================
# Experiment Configuration Area (Modify these three lines as needed)
# Exp1: USE_EARLY_STOP = False, USE_WEIGHTS = False
# Exp2: USE_EARLY_STOP = True,  USE_WEIGHTS = False
# Exp3: USE_EARLY_STOP = True,  USE_WEIGHTS = True
# ==========================================
USE_EARLY_STOP = False
USE_WEIGHTS = False
MAX_EPOCHS = 30
CURRENT_SIZE = 128

seed_everything(42)

# 1. Data Preparation
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, 16)
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)), transforms.ToTensor()
])), batch_size=16, shuffle=False)

# 2. Loss Function and Weight Configuration
if USE_WEIGHTS:
    # Calculate weights in index order: [apple:75, banana:71, mixed:22, orange:72]
    # Logic: Weight = Total samples / (Number of classes * Samples in the class)
    counts = torch.tensor([75, 71, 22, 72], dtype=torch.float)
    weights = (counts.sum() / (4 * counts)).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    print(f" Weighted mode enabled: {weights.cpu().numpy()}")
else:
    criterion = nn.CrossEntropyLoss()
    print(" Standard mode enabled (equal weighting)")

# 3. Initialization
model = FinalFruitCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
early_stopping = EarlyStopping(patience=7)

best_val_acc = 0.0
best_epoch = 0
best_model_wts = None

print(f" Experiment Starts: (EarlyStop={USE_EARLY_STOP}, Weighted={USE_WEIGHTS})")

for epoch in range(MAX_EPOCHS):
    t_acc, t_loss = train_one_epoch_full(model, train_loader, criterion, optimizer, device)
    v_acc, v_loss = evaluate_full(model, val_loader, criterion, device)

    # Save model weights with the highest validation accuracy
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:02d}] Train_Loss: {t_loss:.4f} | Val_Loss: {v_loss:.4f} | Val_Acc: {v_acc:.4f}")

    if USE_EARLY_STOP:
        early_stopping(v_loss)
        if early_stopping.early_stop:
            print(f" [Early Stop Trigger] Model stops at the {epoch+1}th epoch，the optimal epoch is: {best_epoch}")
            break

# 4. Final Testing and Result Presentation
model.load_state_dict(best_model_wts)
test_acc, _ = evaluate_full(model, test_loader, criterion, device)

print("-" * 30)
print(f" Result Summary:")
print(f" Best Epoch: {best_epoch}")
print(f"Peak Val Acc: {best_val_acc:.4f}")
print(f"Final Test Acc: {test_acc:.4f}")

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


 Standard mode enabled (equal weighting)
 Experiment Starts: (EarlyStop=False, Weighted=False)


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01] Train_Loss: 1.3530 | Val_Loss: 1.2679 | Val_Acc: 0.3542
Epoch [02] Train_Loss: 1.2597 | Val_Loss: 1.1433 | Val_Acc: 0.7292
Epoch [03] Train_Loss: 1.0010 | Val_Loss: 0.9293 | Val_Acc: 0.6667
Epoch [04] Train_Loss: 0.7488 | Val_Loss: 0.6261 | Val_Acc: 0.7708
Epoch [05] Train_Loss: 0.5211 | Val_Loss: 0.5581 | Val_Acc: 0.8333
Epoch [06] Train_Loss: 0.3721 | Val_Loss: 0.6017 | Val_Acc: 0.8542
Epoch [07] Train_Loss: 0.3124 | Val_Loss: 0.5278 | Val_Acc: 0.8542
Epoch [08] Train_Loss: 0.2392 | Val_Loss: 0.6066 | Val_Acc: 0.8333
Epoch [09] Train_Loss: 0.2236 | Val_Loss: 0.7401 | Val_Acc: 0.8125
Epoch [10] Train_Loss: 0.2733 | Val_Loss: 0.7705 | Val_Acc: 0.8333
Epoch [11] Train_Loss: 0.1991 | Val_Loss: 0.5877 | Val_Acc: 0.9167
Epoch [12] Train_Loss: 0.1876 | Val_Loss: 0.8254 | Val_Acc: 0.8750
Epoch [13] Train_Loss: 0.1436 | Val_Loss: 0.6931 | Val_Acc: 0.9167
Epoch [14] Train_Loss: 0.1439 | Val_Loss: 0.6241 | Val_Acc: 0.9167
Epoch [15] Train_Loss: 0.1055 | Val_Loss: 0.6260 | Val_Acc: 0.

In [ ]:
def plot_confusion_matrix(model, loader, device, classes):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, lbs in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbs.cpu().numpy())

    # Calculate confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)

    # Plot the confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title('Confusion Matrix: Experiment 1')
    plt.ylabel('True Label (Actual)')
    plt.xlabel('Predicted Label')
    plt.show()

# Class order must be consistent with your Dataset
class_names = ['apple', 'banana', 'orange', 'mixed']

# Run the plotting function directly
plot_confusion_matrix(model, test_loader, device, class_names)

128*128 + 3 layer + dropout 0.3 + early stop

In [ ]:
USE_EARLY_STOP = True
USE_WEIGHTS = False
MAX_EPOCHS = 30
CURRENT_SIZE = 128

seed_everything(42)

# 1. Data Preparation
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, 16)
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)), transforms.ToTensor()
])), batch_size=16, shuffle=False)

# 2. Loss Function and Weight Configuration
if USE_WEIGHTS:
    # Calculate weights in index order: [apple:75, banana:71, mixed:22, orange:72]
    # Logic: Weight = Total samples / (Number of classes * Samples in the class)
    counts = torch.tensor([75, 71, 22, 72], dtype=torch.float)
    weights = (counts.sum() / (4 * counts)).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    print(f" Weighted mode enabled: {weights.cpu().numpy()}")
else:
    criterion = nn.CrossEntropyLoss()
    print(" Standard mode enabled (equal weighting)")

# 3. Initialization
model = FinalFruitCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
early_stopping = EarlyStopping(patience=7)

best_val_acc = 0.0
best_epoch = 0
best_model_wts = None

print(f" Experiment Starts: (EarlyStop={USE_EARLY_STOP}, Weighted={USE_WEIGHTS})")

for epoch in range(MAX_EPOCHS):
    t_acc, t_loss = train_one_epoch_full(model, train_loader, criterion, optimizer, device)
    v_acc, v_loss = evaluate_full(model, val_loader, criterion, device)

    # Save model weights with the highest validation accuracy
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:02d}] Train_Loss: {t_loss:.4f} | Val_Loss: {v_loss:.4f} | Val_Acc: {v_acc:.4f}")

    if USE_EARLY_STOP:
        early_stopping(v_loss)
        if early_stopping.early_stop:
            print(f" [Early Stop Trigger] Model stops at the {epoch+1}th epoch，the optimal epoch is: {best_epoch}")
            break

# 4. Final Testing and Result Presentation
model.load_state_dict(best_model_wts)
test_acc, _ = evaluate_full(model, test_loader, criterion, device)

print("-" * 30)
print(f" Result Summary:")
print(f" Best Epoch: {best_epoch}")
print(f"Peak Val Acc: {best_val_acc:.4f}")
print(f"Final Test Acc: {test_acc:.4f}")

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


 Standard mode enabled (equal weighting)
 Experiment Starts: (EarlyStop=True, Weighted=False)


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01] Train_Loss: 1.3530 | Val_Loss: 1.2679 | Val_Acc: 0.3542
Epoch [02] Train_Loss: 1.2597 | Val_Loss: 1.1433 | Val_Acc: 0.7292
Epoch [03] Train_Loss: 1.0010 | Val_Loss: 0.9293 | Val_Acc: 0.6667
Epoch [04] Train_Loss: 0.7488 | Val_Loss: 0.6261 | Val_Acc: 0.7708
Epoch [05] Train_Loss: 0.5211 | Val_Loss: 0.5581 | Val_Acc: 0.8333
Epoch [06] Train_Loss: 0.3721 | Val_Loss: 0.6017 | Val_Acc: 0.8542
Epoch [07] Train_Loss: 0.3124 | Val_Loss: 0.5278 | Val_Acc: 0.8542
Epoch [08] Train_Loss: 0.2392 | Val_Loss: 0.6066 | Val_Acc: 0.8333
Epoch [09] Train_Loss: 0.2236 | Val_Loss: 0.7401 | Val_Acc: 0.8125
Epoch [10] Train_Loss: 0.2733 | Val_Loss: 0.7705 | Val_Acc: 0.8333
Epoch [11] Train_Loss: 0.1991 | Val_Loss: 0.5877 | Val_Acc: 0.9167
Epoch [12] Train_Loss: 0.1876 | Val_Loss: 0.8254 | Val_Acc: 0.8750
Epoch [13] Train_Loss: 0.1436 | Val_Loss: 0.6931 | Val_Acc: 0.9167
Epoch [14] Train_Loss: 0.1439 | Val_Loss: 0.6241 | Val_Acc: 0.9167
 [Early Stop Trigger] Model stops at the 14th epoch，the optima

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
import pandas as pd

def plot_confusion_matrix(model, loader, device, classes):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, lbs in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbs.cpu().numpy())

    # Calculate confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)

    # Plot the confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title('Confusion Matrix: Experiment 2 (early stop)')
    plt.ylabel('True Label (Actual)')
    plt.xlabel('Predicted Label')
    plt.show()

# Class order must be consistent with your Dataset
class_names = ['apple', 'banana', 'orange', 'mixed']

# Run the plotting function directly
plot_confusion_matrix(model, test_loader, device, class_names)

128*128 + 3 layer + dropout 0.3 + early stop + weighted loss

In [ ]:
USE_EARLY_STOP = True
USE_WEIGHTS = True
MAX_EPOCHS = 30
CURRENT_SIZE = 128

seed_everything(42)

# 1. Data Preparation
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, 16)
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)), transforms.ToTensor()
])), batch_size=16, shuffle=False)

# 2. Loss Function and Weight Configuration
if USE_WEIGHTS:
    # Calculate class weights in index order: [apple:75, banana:71, mixed:22, orange:72]
    # Logic: Weight = Total samples / (Number of classes * Samples in the class)
    counts = torch.tensor([75, 71, 22, 72], dtype=torch.float)
    weights = (counts.sum() / (4 * counts)).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    print(f" Weighted mode enabled: {weights.cpu().numpy()}")
else:
    criterion = nn.CrossEntropyLoss()
    print(" Standard mode enabled (equal weighting)")

# 3. Initialization
model = FinalFruitCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
early_stopping = EarlyStopping(patience=7)

best_val_acc = 0.0
best_epoch = 0
best_model_wts = None

print(f" Experiment Starts: (EarlyStop={USE_EARLY_STOP}, Weighted={USE_WEIGHTS})")

for epoch in range(MAX_EPOCHS):
    t_acc, t_loss = train_one_epoch_full(model, train_loader, criterion, optimizer, device)
    v_acc, v_loss = evaluate_full(model, val_loader, criterion, device)

    # Save model weights with the highest validation accuracy
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:02d}] Train_Loss: {t_loss:.4f} | Val_Loss: {v_loss:.4f} | Val_Acc: {v_acc:.4f}")

    if USE_EARLY_STOP:
        early_stopping(v_loss)
        if early_stopping.early_stop:
            print(f" [Early Stop Trigger] Model stops at epoch {epoch+1}, optimal epoch is: {best_epoch}")
            break

# 4. Final Testing and Result Presentation
model.load_state_dict(best_model_wts)
test_acc, _ = evaluate_full(model, test_loader, criterion, device)

print("-" * 30)
print(f" Result Summary:")
print(f" Best Epoch: {best_epoch}")
print(f"Peak Val Acc: {best_val_acc:.4f}")
print(f"Final Test Acc: {test_acc:.4f}")

 Weighted mode enabled: [0.8       0.8450704 2.7272727 0.8333333]
 Experiment Starts: (EarlyStop=True, Weighted=True)


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01] Train_Loss: 1.2483 | Val_Loss: 1.0618 | Val_Acc: 0.3125
Epoch [02] Train_Loss: 1.1029 | Val_Loss: 1.0368 | Val_Acc: 0.3125
Epoch [03] Train_Loss: 1.0649 | Val_Loss: 0.9826 | Val_Acc: 0.3125
Epoch [04] Train_Loss: 1.0035 | Val_Loss: 0.8641 | Val_Acc: 0.3125
Epoch [05] Train_Loss: 0.7979 | Val_Loss: 0.6460 | Val_Acc: 0.6875
Epoch [06] Train_Loss: 0.5372 | Val_Loss: 0.5497 | Val_Acc: 0.7500
Epoch [07] Train_Loss: 0.4403 | Val_Loss: 0.4739 | Val_Acc: 0.8750
Epoch [08] Train_Loss: 0.3281 | Val_Loss: 0.5546 | Val_Acc: 0.8333
Epoch [09] Train_Loss: 0.2996 | Val_Loss: 0.5838 | Val_Acc: 0.8542
Epoch [10] Train_Loss: 0.3014 | Val_Loss: 0.4262 | Val_Acc: 0.8958
Epoch [11] Train_Loss: 0.2470 | Val_Loss: 0.4727 | Val_Acc: 0.8750
Epoch [12] Train_Loss: 0.2606 | Val_Loss: 0.6402 | Val_Acc: 0.8542
Epoch [13] Train_Loss: 0.2420 | Val_Loss: 0.4717 | Val_Acc: 0.9167
Epoch [14] Train_Loss: 0.1821 | Val_Loss: 0.4409 | Val_Acc: 0.8958
Epoch [15] Train_Loss: 0.1399 | Val_Loss: 0.4343 | Val_Acc: 0.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
import pandas as pd

def plot_confusion_matrix(model, loader, device, classes):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, lbs in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbs.cpu().numpy())

    # Calculate confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)

    # Plot the confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title('Confusion Matrix: Experiment 3 (Weighted)')
    plt.ylabel('True Label (Actual)')
    plt.xlabel('Predicted Label')
    plt.show()

# The class order must be consistent with your Dataset
class_names = ['apple', 'banana', 'orange', 'mixed']

# Run the plotting function directly
plot_confusion_matrix(model, test_loader, device, class_names)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix

# ================= Data Simulation Area =================
# Exp1: No ES, No Weight ; Exp2: ES, No Weight ; Exp3: ES, Weighted
exp_names = ['Exp1: Baseline', 'Exp2: +EarlyStop', 'Exp3: +Weighted']

# 1. Record the Loss curve of Experiment 1 to demonstrate overfitting
train_loss_exp1 = [1.3530, 1.2597, 1.0010, 0.7488, 0.5211, 0.3721, 0.3124, 0.2392, 0.2236, 0.2733, 0.1991, 0.1876, 0.1436, 0.1439, 0.1055, 0.1103, 0.0846, 0.0986, 0.1577, 0.1273, 0.1597, 0.1231, 0.0675, 0.0465, 0.0392, 0.1067, 0.1235, 0.0718, 0.0700, 0.0563]
val_loss_exp1 = [1.2679, 1.1433, 0.9293, 0.6261, 0.5581, 0.6017, 0.5278, 0.6066, 0.7401, 0.7705, 0.5877, 0.8254, 0.6931, 0.6241, 0.6260, 0.6684, 0.6765, 1.2492, 0.7517, 0.6739, 0.6472, 0.7514, 0.7284, 0.5995, 0.9597, 0.7329, 0.7480, 0.6111, 1.0224, 0.7439]

# 2. Accuracy Summary
best_val_accs = [0.9375, 0.9167, 0.9167]
final_test_accs = [0.9000, 0.8667, 0.8500]


# ================= Plotting Area =================
plt.figure(figsize=(20, 6))

# Plot 1: Loss Curve of Experiment 1 —— Demonstrate Overfitting and the Necessity of Early Stopping
plt.subplot(1, 3, 1)
epochs = range(1, 31)
plt.plot(epochs, train_loss_exp1, 'b-o', label='Train Loss', markersize=4)
plt.plot(epochs, val_loss_exp1, 'r-o', label='Val Loss', markersize=4)
plt.axvline(x=7, color='gray', linestyle='--', label='Min Val Loss (Ep 7)')
plt.axvline(x=14, color='orange', linestyle='-', label='Early Stop Point')
plt.annotate('Overfitting Starts', xy=(8, 0.8), xytext=(10, 1.2),
             arrowprops=dict(facecolor='black', shrink=0.05))
plt.title(' Overfitting Evidence ')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Plot 2: Accuracy Comparison of Three Experiments —— Generalization Ability Analysis
plt.subplot(1, 3, 2)
x = np.arange(len(exp_names))
width = 0.35
plt.bar(x - width/2, best_val_accs, width, label='Best Val Acc', color='skyblue')
plt.bar(x + width/2, final_test_accs, width, label='Final Test Acc', color='salmon')
plt.xticks(x, exp_names)
plt.ylim(0.8, 1.0) # Zoom in on the difference area
plt.ylabel('Accuracy')
plt.title('Accuracy Comparison')
plt.legend()


plt.tight_layout()
plt.show()

Transformation Comparison: Original vs Resize vs CenterCrop

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random
import os
from torchvision import transforms
from matplotlib.lines import Line2D # Import Line2D

def visualize_transform_comparison_v2(root_dir, target_size=(128, 128)):
    class_map = {"apple": 0, "banana": 1, "orange": 2, "mixed": 3}
    all_files = [f for f in os.listdir(root_dir) if f.lower().endswith(".jpg")]

    category_files = {cls: [] for cls in class_map.keys()}
    for f in all_files:
        for cls in class_map.keys():
            if f.lower().startswith(cls):
                category_files[cls].append(f)
                break

    # Define transformations
    resize_tf = transforms.Resize(target_size)
    crop_tf = transforms.Compose([
        transforms.Resize(140),
        transforms.CenterCrop(target_size)
    ])

    # Create plot canvas
    num_cats = len(class_map)
    fig, axes = plt.subplots(num_cats, 3, figsize=(15, 4 * num_cats))
    plt.suptitle("Transformation Comparison: Original vs Resize vs CenterCrop", fontsize=20, y=0.95)

    for i, (cls, files) in enumerate(category_files.items()):
        if not files: continue

        img_name = random.choice(files)
        img_path = os.path.join(root_dir, img_name)
        orig_img = Image.open(img_path).convert('RGB')

        imgs = [orig_img, resize_tf(orig_img), crop_tf(orig_img)]
        titles = [f"Original: {cls}\n({orig_img.size[0]}x{orig_img.size[1]})",
                  "Resize (Distorted)\n128x128",
                  "CenterCrop (Fixed)\n128x128"]
        colors = ['black', 'red', 'green']

        for j in range(3):
            axes[i, j].imshow(imgs[j])
            axes[i, j].set_title(titles[j], fontsize=11, color=colors[j])
            axes[i, j].axis('off')

            # Key modification: Force the display frame to adapt to the image aspect ratio without automatic stretching
            axes[i, j].set_anchor('C')

    # Automatically adjust subplot spacing
    plt.tight_layout(rect=[0, 0.03, 1, 0.93])

    plt.show()

# Run the function
visualize_transform_comparison_v2(TRAIN_DIR)

In [ ]:
from collections import Counter
def get_label_from_filename(filename):
    return filename.split("_")[0].lower()

train_files = os.listdir(TRAIN_DIR)
train_labels = [get_label_from_filename(f) for f in train_files]

class_counts = Counter(train_labels)
class_counts

Counter({'apple': 75, 'banana': 71, 'orange': 72, 'mixed': 70})

In [ ]:
classes = list(class_counts.keys())
counts = list(class_counts.values())

plt.figure(figsize=(6, 5))
bars=plt.bar(classes, counts,width=0.5)
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.title("Class Distribution in Training Set")
plt.bar_label(bars, padding=2)
plt.tight_layout()

plt.savefig("class_distribution_train.png", dpi=300)
plt.show()

In [ ]:
def get_loaders(root_dir, img_size, batch_size):
    # 1. Fix: Add normalization to keep consistency with previous experiments
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
    ])

    full_dataset = FruitDataset(root_dir, transform=transform)
    indices = list(range(len(full_dataset)))

    # 2. Optimization: Extract labels directly from filenames without triggering image reading (assuming FruitDataset has the image_files attribute)
    # If this attribute does not exist, ensure your dataset class has a list to store file paths
    labels = []
    for img_path in full_dataset.image_files:
        filename = os.path.basename(img_path).lower()
        if filename.startswith("apple"): labels.append(0)
        elif filename.startswith("banana"): labels.append(1)
        elif filename.startswith("orange"): labels.append(2)
        elif filename.startswith("mixed"): labels.append(3)

    # 3. Stratified sampling
    train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=labels, random_state=42)

    train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(full_dataset, val_idx), batch_size=batch_size, shuffle=False)

    return train_loader, val_loader

In [ ]:
USE_EARLY_STOP = True
USE_WEIGHTS = False
MAX_EPOCHS = 30
CURRENT_SIZE = 128

seed_everything(42)

# 1. Data preparation
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, 16)
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)), transforms.ToTensor(),transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])), batch_size=16, shuffle=False)

# 2. Loss function and weight configuration
if USE_WEIGHTS:
    # Calculate weights in index order: [apple:75, banana:71, mixed:70, orange:72]
    # Logic: Weight = Total number of samples / (Number of classes * Number of samples in the class)
    counts = torch.tensor([75, 71, 70, 72], dtype=torch.float)
    weights = (counts.sum() / (4 * counts)).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    print(f" Weighted mode enabled: {weights.cpu().numpy()}")
else:
    criterion = nn.CrossEntropyLoss()
    print(" Standard mode enabled (equal weighting)")

# 3. Initialization
model = FinalFruitCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
early_stopping = EarlyStopping(patience=7)

best_val_acc = 0.0
best_epoch = 0
best_model_wts = copy.deepcopy(model.state_dict())

print(f" Experiment Starts: (EarlyStop={USE_EARLY_STOP}, Weighted={USE_WEIGHTS})")

for epoch in range(MAX_EPOCHS):
    t_acc, t_loss = train_one_epoch_full(model, train_loader, criterion, optimizer, device)
    v_acc, v_loss = evaluate_full(model, val_loader, criterion, device)

    # Save the model weights with the highest accuracy
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())

    print(f"Epoch [{epoch+1:02d}] Train_Loss: {t_loss:.4f} | Val_Loss: {v_loss:.4f} | Val_Acc: {v_acc:.4f}")

    if USE_EARLY_STOP:
        early_stopping(v_loss)
        if early_stopping.early_stop:
            print(f" [Early Stop Trigger] Model stops at the {epoch+1}th epoch, the optimal epoch is: {best_epoch}")
            break

# 4. Final test and result display
model.load_state_dict(best_model_wts)
test_acc, _ = evaluate_full(model, test_loader, criterion, device)

print("-" * 30)
print(f" Result Summary:")
print(f" Best Epoch: {best_epoch}")
print(f"Peak Val Acc: {best_val_acc:.4f}")
print(f"Final Test Acc: {test_acc:.4f}")

 Standard mode enabled (equal weighting)
 Experiment Starts: (EarlyStop=True, Weighted=False)


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01] Train_Loss: 1.3276 | Val_Loss: 1.1548 | Val_Acc: 0.6034
Epoch [02] Train_Loss: 0.9428 | Val_Loss: 1.1319 | Val_Acc: 0.5517
Epoch [03] Train_Loss: 0.8369 | Val_Loss: 0.7717 | Val_Acc: 0.6724
Epoch [04] Train_Loss: 0.6621 | Val_Loss: 0.7670 | Val_Acc: 0.6552
Epoch [05] Train_Loss: 0.5448 | Val_Loss: 0.6684 | Val_Acc: 0.7414
Epoch [06] Train_Loss: 0.3755 | Val_Loss: 0.5541 | Val_Acc: 0.8276
Epoch [07] Train_Loss: 0.3541 | Val_Loss: 0.5829 | Val_Acc: 0.8103
Epoch [08] Train_Loss: 0.2396 | Val_Loss: 0.5895 | Val_Acc: 0.8276
Epoch [09] Train_Loss: 0.1840 | Val_Loss: 0.6574 | Val_Acc: 0.8621
Epoch [10] Train_Loss: 0.1829 | Val_Loss: 0.8767 | Val_Acc: 0.7414
Epoch [11] Train_Loss: 0.1755 | Val_Loss: 0.6987 | Val_Acc: 0.8276
Epoch [12] Train_Loss: 0.1709 | Val_Loss: 0.6121 | Val_Acc: 0.8621
Epoch [13] Train_Loss: 0.1635 | Val_Loss: 0.7345 | Val_Acc: 0.7931
 [Early Stop Trigger] Model stops at the 13th epoch，the optimal epoch is: 9
------------------------------
 Result Summary:
 Best

In [ ]:
def plot_confusion_matrix(model, loader, device, classes):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, lbs in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbs.cpu().numpy())

    # Calculate confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)

    # Plot the confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title('Confusion Matrix: Experiment 4 (Balanced Data & Early Stop)')
    plt.ylabel('True Label (Actual)')
    plt.xlabel('Predicted Label')
    plt.show()

# The class order must be consistent with your Dataset
class_names = ['apple', 'banana', 'orange', 'mixed']

# Run the plotting function directly
plot_confusion_matrix(model, test_loader, device, class_names)

In [ ]:
import copy

USE_EARLY_STOP = True
MAX_EPOCHS = 50
CURRENT_SIZE = 128

seed_everything(42)

# 1. Data Preparation (Core Modification: Weakened Augmentation Strategy)

# Fix: Define weakened augmentation version v5_v2
train_transform_v5_v2 = transforms.Compose([
    # Increase scale to 0.85-1.0 to prevent fruits from being cropped to only background
    transforms.RandomResizedCrop(CURRENT_SIZE, scale=(0.85, 1.0), ratio=(1.0, 1.0)),
    # Keep the most robust flip operation
    transforms.RandomHorizontalFlip(p=0.5),
    # Temporarily remove rotation and color jitter to reduce the burden on the simple model
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

standard_transform = transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# Fix: Ensure train_transform_v5_v2 is referenced here
full_dataset_train = FruitDataset(TRAIN_DIR, transform=train_transform_v5_v2)
full_dataset_val = FruitDataset(TRAIN_DIR, transform=standard_transform)

indices = list(range(len(full_dataset_train)))
# Extract labels for stratified sampling
labels = [0 if "apple" in f.lower() else 1 if "banana" in f.lower() else 2 if "orange" in f.lower() else 3 for f in full_dataset_train.image_files]
train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=labels, random_state=42)

train_loader = DataLoader(Subset(full_dataset_train, train_idx), batch_size=16, shuffle=True)
val_loader = DataLoader(Subset(full_dataset_val, val_idx), batch_size=16, shuffle=False)
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=standard_transform), batch_size=16, shuffle=False)

#  2. Loss Function
criterion = nn.CrossEntropyLoss()

# 3. Initialization and Scheduler
model = FinalFruitCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Fix: Increase patience to 6 to give the model more time to adapt to "altered" images
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=6)
early_stopping = EarlyStopping(patience=10) # Increase early stopping patience accordingly

best_val_acc = 0.0
best_epoch = 0
best_model_wts = copy.deepcopy(model.state_dict())

print(f" Experiment 5: Data Augmentation ")

for epoch in range(MAX_EPOCHS):
    t_acc, t_loss = train_one_epoch_full(model, train_loader, criterion, optimizer, device)
    v_acc, v_loss = evaluate_full(model, val_loader, criterion, device)

    # Step the scheduler
    scheduler.step(v_loss)

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())

    # Print current learning rate for observation
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1:02d}] T_Loss: {t_loss:.4f} | V_Loss: {v_loss:.4f} | V_Acc: {v_acc:.4f} | LR: {current_lr:.6f}")

    if USE_EARLY_STOP:
        early_stopping(v_loss)
        if early_stopping.early_stop:
            print(f" [Early Stop] Optimal epoch: {best_epoch}")
            break

# 4. Final Testing
model.load_state_dict(best_model_wts)
test_acc, _ = evaluate_full(model, test_loader, criterion, device)

print("-" * 30)
print(f"Final Result Summary:")
print(f"Best Epoch: {best_epoch} | Peak Val Acc: {best_val_acc:.4f} | Final Test Acc: {test_acc:.4f}")

 Experiment 5: Data Augmentation 


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01] T_Loss: 1.3097 | V_Loss: 1.1510 | V_Acc: 0.6724 | LR: 0.001000
Epoch [02] T_Loss: 0.9711 | V_Loss: 0.9618 | V_Acc: 0.5517 | LR: 0.001000
Epoch [03] T_Loss: 0.7911 | V_Loss: 0.7468 | V_Acc: 0.7414 | LR: 0.001000
Epoch [04] T_Loss: 0.5774 | V_Loss: 0.8414 | V_Acc: 0.6552 | LR: 0.001000
Epoch [05] T_Loss: 0.5314 | V_Loss: 0.7081 | V_Acc: 0.7414 | LR: 0.001000
Epoch [06] T_Loss: 0.4059 | V_Loss: 0.6320 | V_Acc: 0.8621 | LR: 0.001000
Epoch [07] T_Loss: 0.2776 | V_Loss: 1.1500 | V_Acc: 0.7069 | LR: 0.001000
Epoch [08] T_Loss: 0.4343 | V_Loss: 0.7393 | V_Acc: 0.8103 | LR: 0.001000
Epoch [09] T_Loss: 0.2533 | V_Loss: 0.7187 | V_Acc: 0.8276 | LR: 0.001000
Epoch [10] T_Loss: 0.2668 | V_Loss: 0.7487 | V_Acc: 0.8621 | LR: 0.001000
Epoch [11] T_Loss: 0.2225 | V_Loss: 0.7660 | V_Acc: 0.8621 | LR: 0.001000
Epoch [12] T_Loss: 0.2048 | V_Loss: 0.6885 | V_Acc: 0.9138 | LR: 0.001000
Epoch [13] T_Loss: 0.1713 | V_Loss: 0.7390 | V_Acc: 0.8793 | LR: 0.000100
Epoch [14] T_Loss: 0.1463 | V_Loss: 0.

In [ ]:
def plot_confusion_matrix(model, loader, device, classes):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, lbs in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbs.cpu().numpy())

    # Calculate confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)

    # Plot the confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title('Confusion Matrix: Experiment 5 (Balanced Data & Early Stop)')
    plt.ylabel('True Label (Actual)')
    plt.xlabel('Predicted Label')
    plt.show()

# The class order must be consistent with your Dataset
class_names = ['apple', 'banana', 'orange', 'mixed']

# Run the plotting function directly

In [ ]:
USE_EARLY_STOP = True
MAX_EPOCHS = 50
CURRENT_SIZE = 128
seed_everything(42)

# 1. Data Preparation (Core Modification: Augmentation Strategy for Orange Misclassification)

train_transform_v6 = transforms.Compose([
    # Strategy A: Contour Protection. Increase the lower crop limit to 0.9 to ensure the model can see the circular edge of oranges in most cases
    # Avoid misclassification as Mixed due to "only focusing on local texture"
    transforms.RandomResizedCrop(CURRENT_SIZE, scale=(0.92, 1.0), ratio=(0.95, 1.05)),

    # Strategy B: Horizontal Flip (standard operation)
    transforms.RandomHorizontalFlip(p=0.5),

    # Strategy C: Color Perturbation (key regression!)
    # We add slight changes to hue and saturation.
    # This tells the model: "Don't just look at color, look at shape!" because Mixed also has orange color.
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.2, hue=0.05),

    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# Validation and Test Sets: Keep standard Resize
standard_transform = transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

# Load data
full_dataset_train = FruitDataset(TRAIN_DIR, transform=train_transform_v6)
full_dataset_val = FruitDataset(TRAIN_DIR, transform=standard_transform)

indices = list(range(len(full_dataset_train)))
labels = [0 if "apple" in f.lower() else 1 if "banana" in f.lower() else 2 if "orange" in f.lower() else 3 for f in full_dataset_train.image_files]
train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=labels, random_state=42)

train_loader = DataLoader(Subset(full_dataset_train, train_idx), batch_size=16, shuffle=True)
val_loader = DataLoader(Subset(full_dataset_val, val_idx), batch_size=16, shuffle=False)
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=standard_transform), batch_size=16, shuffle=False)

#2. Loss Function
criterion = nn.CrossEntropyLoss()

# 3. Initialization and Scheduler
model = FinalFruitCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Slightly reduce patience to make learning rate changes more sensitive, helping the model escape local optima
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)
early_stopping = EarlyStopping(patience=8)

best_val_acc = 0.0
best_epoch = 0
best_model_wts = copy.deepcopy(model.state_dict())

print(f" Experiment 6 Starts: Contour Protection + Color Robustness")

for epoch in range(MAX_EPOCHS):
    t_acc, t_loss = train_one_epoch_full(model, train_loader, criterion, optimizer, device)
    v_acc, v_loss = evaluate_full(model, val_loader, criterion, device)

    scheduler.step(v_loss)

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1:02d}] T_Loss: {t_loss:.4f} | V_Loss: {v_loss:.4f} | V_Acc: {v_acc:.4f} | LR: {current_lr:.6f}")

    if USE_EARLY_STOP:
        early_stopping(v_loss)
        if early_stopping.early_stop:
            print(f" [Early Stop] Optimal epoch: {best_epoch}")
            break

#4. Final Testing
model.load_state_dict(best_model_wts)
test_acc, _ = evaluate_full(model, test_loader, criterion, device)

print("-" * 30)
print(f"Final Result Summary:")
print(f"Best Epoch: {best_epoch} | Peak Val Acc: {best_val_acc:.4f} | Final Test Acc: {test_acc:.4f}")

 Experiment 6 Starts: Contour Protection + Color Robustness


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01] T_Loss: 1.3242 | V_Loss: 1.1604 | V_Acc: 0.6724 | LR: 0.001000
Epoch [02] T_Loss: 1.0685 | V_Loss: 0.9390 | V_Acc: 0.6552 | LR: 0.001000
Epoch [03] T_Loss: 0.9012 | V_Loss: 0.7321 | V_Acc: 0.7241 | LR: 0.001000
Epoch [04] T_Loss: 0.7435 | V_Loss: 0.7940 | V_Acc: 0.6379 | LR: 0.001000
Epoch [05] T_Loss: 0.7544 | V_Loss: 0.6739 | V_Acc: 0.7241 | LR: 0.001000
Epoch [06] T_Loss: 0.6305 | V_Loss: 0.5882 | V_Acc: 0.7759 | LR: 0.001000
Epoch [07] T_Loss: 0.6051 | V_Loss: 0.6078 | V_Acc: 0.7241 | LR: 0.001000
Epoch [08] T_Loss: 0.5812 | V_Loss: 0.5649 | V_Acc: 0.8448 | LR: 0.001000
Epoch [09] T_Loss: 0.5901 | V_Loss: 0.7354 | V_Acc: 0.7241 | LR: 0.001000
Epoch [10] T_Loss: 0.6467 | V_Loss: 0.8057 | V_Acc: 0.7241 | LR: 0.001000
Epoch [11] T_Loss: 0.5251 | V_Loss: 0.5618 | V_Acc: 0.8276 | LR: 0.001000
Epoch [12] T_Loss: 0.5119 | V_Loss: 0.5570 | V_Acc: 0.8448 | LR: 0.001000
Epoch [13] T_Loss: 0.4858 | V_Loss: 0.7079 | V_Acc: 0.7586 | LR: 0.001000
Epoch [14] T_Loss: 0.4855 | V_Loss: 0.

In [ ]:

USE_EARLY_STOP = True
MAX_EPOCHS = 40          # Cosine annealing usually performs better within a fixed cycle
BATCH_SIZE = 32          # Optimization: Increase Batch Size for more stable gradient statistics
CURRENT_SIZE = 128
LR = 0.001

seed_everything(42)

#  Label Smoothing Loss
# Key solution to overfitting caused by overlapping between Orange and Mixed classes
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes, smoothing=0.1):
        super(LabelSmoothingLoss, self).__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes

    def forward(self, pred, target):
        pred = pred.log_softmax(dim=-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.cls - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        return torch.mean(torch.sum(-true_dist * pred, dim=-1))

#  Data Augmentation Strategy (Shape-Preserving Translation)
# Abandon ResizedCrop to prevent banana distortion, use Resize + RandomCrop instead
train_transform_optimized = transforms.Compose([
    transforms.Resize((140, 140)),       # Enlarge first
    transforms.RandomCrop(CURRENT_SIZE), # Random crop to target size (creates translation effect without changing aspect ratio)
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

standard_transform = transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

#  Data Loading & Stratified Sampling
full_dataset_train = FruitDataset(TRAIN_DIR, transform=train_transform_optimized)
full_dataset_val = FruitDataset(TRAIN_DIR, transform=standard_transform)

indices = list(range(len(full_dataset_train)))
labels = [0 if "apple" in f.lower() else 1 if "banana" in f.lower() else 2 if "orange" in f.lower() else 3 for f in full_dataset_train.image_files]
train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=labels, random_state=42)

train_loader = DataLoader(Subset(full_dataset_train, train_idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(Subset(full_dataset_val, val_idx), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=standard_transform), batch_size=BATCH_SIZE, shuffle=False)

#  Model Initialization, Optimizer & Scheduler
model = FinalFruitCNN().to(device)

# Optimization: Switch to AdamW and add weight decay to prevent overfitting
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# Optimization: Adopt Cosine Annealing learning rate strategy
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

# Optimization: Use Label Smoothing Loss
criterion = LabelSmoothingLoss(classes=4, smoothing=0.1)

early_stopping = EarlyStopping(patience=12) # Leave enough time for cosine annealing

best_val_acc = 0.0
best_epoch = 0
best_model_wts = copy.deepcopy(model.state_dict())

print(f" Starting Optimized Experiment : [BatchSize={BATCH_SIZE}, AdamW, CosineLR, LabelSmoothing]")

#  Training Loop
for epoch in range(MAX_EPOCHS):
    # Training phase
    model.train()
    train_running_loss = 0.0
    train_correct = 0
    train_total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

    t_loss = train_running_loss / len(train_idx)
    t_acc = train_correct / train_total

    # Validation phase
    v_acc, v_loss = evaluate_full(model, val_loader, criterion, device)

    # Scheduler step (cosine annealing updates automatically by epoch)
    scheduler.step()

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_epoch = epoch + 1
        best_model_wts = copy.deepcopy(model.state_dict())

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1:02d}] T_Loss: {t_loss:.4f} | V_Loss: {v_loss:.4f} | V_Acc: {v_acc:.4f} | LR: {current_lr:.6f}")

    if USE_EARLY_STOP:
        early_stopping(v_loss)
        if early_stopping.early_stop:
            print(f" [Early Stop Triggered] Optimal epoch: {best_epoch}")
            break

#  Final Evaluation
model.load_state_dict(best_model_wts)
test_acc, _ = evaluate_full(model, test_loader, criterion, device)

print("-" * 30)
print(f" Optimized Result Summary:")
print(f"Best Epoch: {best_epoch} | Peak Val Acc: {best_val_acc:.4f} | Final Test Acc: {test_acc:.4f}")

 Starting Optimized Experiment : [BatchSize=32, AdamW, CosineLR, LabelSmoothing]


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01] T_Loss: 1.3672 | V_Loss: 1.2868 | V_Acc: 0.4828 | LR: 0.000998
Epoch [02] T_Loss: 1.2460 | V_Loss: 1.1829 | V_Acc: 0.4483 | LR: 0.000994
Epoch [03] T_Loss: 1.1068 | V_Loss: 1.1346 | V_Acc: 0.6552 | LR: 0.000986
Epoch [04] T_Loss: 0.9985 | V_Loss: 0.9628 | V_Acc: 0.7759 | LR: 0.000976
Epoch [05] T_Loss: 0.9214 | V_Loss: 0.9430 | V_Acc: 0.7414 | LR: 0.000962
Epoch [06] T_Loss: 0.8404 | V_Loss: 0.8780 | V_Acc: 0.7759 | LR: 0.000946
Epoch [07] T_Loss: 0.7706 | V_Loss: 0.8719 | V_Acc: 0.7759 | LR: 0.000926
Epoch [08] T_Loss: 0.7668 | V_Loss: 0.8081 | V_Acc: 0.8448 | LR: 0.000905
Epoch [09] T_Loss: 0.7303 | V_Loss: 0.8733 | V_Acc: 0.8448 | LR: 0.000880
Epoch [10] T_Loss: 0.7246 | V_Loss: 0.7668 | V_Acc: 0.8448 | LR: 0.000854
Epoch [11] T_Loss: 0.6925 | V_Loss: 0.7725 | V_Acc: 0.8621 | LR: 0.000825
Epoch [12] T_Loss: 0.6879 | V_Loss: 0.7418 | V_Acc: 0.8966 | LR: 0.000794
Epoch [13] T_Loss: 0.6686 | V_Loss: 0.7399 | V_Acc: 0.8966 | LR: 0.000761
Epoch [14] T_Loss: 0.6442 | V_Loss: 0.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from sklearn.model_selection import train_test_split
import copy

# 1. Define Widened Model (Wide-FruitCNN)
class WideFruitCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(WideFruitCNN, self).__init__()
        # Increase width: from 16-32-64 to 32-64-128
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)

        # 128x128 becomes 16x16 after 3 MaxPool operations
        self.fc1 = nn.Linear(128 * 16 * 16, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        x = x.view(-1, 128 * 16 * 16)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# 2. Label Smoothing Loss Function
class LabelSmoothingLoss(nn.Module):
    def __init__(self, classes, smoothing=0.1):
        super(LabelSmoothingLoss, self).__init__()
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.cls = classes

    def forward(self, pred, target):
        pred = pred.log_softmax(dim=-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.cls - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), self.confidence)
        return torch.mean(torch.sum(-true_dist * pred, dim=-1))

# 3. Configuration & Data Preparation
MAX_EPOCHS = 40
BATCH_SIZE = 32
CURRENT_SIZE = 128
LR = 0.001
seed_everything(42)

# Shape-preserving translation augmentation
train_transform = transforms.Compose([
    transforms.Resize((140, 140)),
    transforms.RandomCrop(CURRENT_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

standard_transform = transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

full_dataset_train = FruitDataset(TRAIN_DIR, transform=train_transform)
full_dataset_val = FruitDataset(TRAIN_DIR, transform=standard_transform)

indices = list(range(len(full_dataset_train)))
labels = [0 if "apple" in f.lower() else 1 if "banana" in f.lower() else 2 if "orange" in f.lower() else 3 for f in full_dataset_train.image_files]
train_idx, val_idx = train_test_split(indices, test_size=0.2, stratify=labels, random_state=42)

train_loader = DataLoader(Subset(full_dataset_train, train_idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(Subset(full_dataset_val, val_idx), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=standard_transform), batch_size=BATCH_SIZE, shuffle=False)

# 4. Initialization
model = WideFruitCNN().to(device)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
criterion = LabelSmoothingLoss(classes=4, smoothing=0.1)
early_stopping = EarlyStopping(patience=10)

best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

print(f"🚀 Final Sprint: Wide-FruitCNN with V_Loss Display")

#5. Training Loop
for epoch in range(MAX_EPOCHS):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

    t_loss = train_loss / len(train_idx)
    t_acc = train_correct / train_total

    # Validation phase
    v_acc, v_loss = evaluate_full(model, val_loader, criterion, device)
    scheduler.step()

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_model_wts = copy.deepcopy(model.state_dict())

    # Print log including Val_Loss
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1:02d}] T_Loss: {t_loss:.4f} | V_Loss: {v_loss:.4f} | V_Acc: {v_acc:.4f} | LR: {current_lr:.6f}")

    if early_stopping:
        early_stopping(v_loss)
        if early_stopping.early_stop:
            print(" [Early Stop Triggered]")
            break

#6. Final Testing
model.load_state_dict(best_model_wts)
test_acc, _ = evaluate_full(model, test_loader, criterion, device)

print("-" * 30)
print(f"🏆 Results Summary:")
print(f"Peak Val Acc: {best_val_acc:.4f} | Final Test Acc: {test_acc:.4f}")

🚀 Final Sprint: Wide-FruitCNN with V_Loss Display


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01] T_Loss: 11.1024 | V_Loss: 2.6349 | V_Acc: 0.3276 | LR: 0.000998
Epoch [02] T_Loss: 3.1661 | V_Loss: 1.3795 | V_Acc: 0.6379 | LR: 0.000994
Epoch [03] T_Loss: 1.6906 | V_Loss: 1.1316 | V_Acc: 0.7069 | LR: 0.000986
Epoch [04] T_Loss: 1.1976 | V_Loss: 1.0279 | V_Acc: 0.7759 | LR: 0.000976
Epoch [05] T_Loss: 0.9176 | V_Loss: 0.8614 | V_Acc: 0.8448 | LR: 0.000962
Epoch [06] T_Loss: 0.8274 | V_Loss: 0.8567 | V_Acc: 0.7759 | LR: 0.000946
Epoch [07] T_Loss: 0.8170 | V_Loss: 0.8420 | V_Acc: 0.7931 | LR: 0.000926
Epoch [08] T_Loss: 0.7954 | V_Loss: 0.7614 | V_Acc: 0.9138 | LR: 0.000905
Epoch [09] T_Loss: 0.7697 | V_Loss: 0.7583 | V_Acc: 0.8793 | LR: 0.000880
Epoch [10] T_Loss: 0.7696 | V_Loss: 0.7476 | V_Acc: 0.8793 | LR: 0.000854
Epoch [11] T_Loss: 0.7230 | V_Loss: 0.7818 | V_Acc: 0.8793 | LR: 0.000825
Epoch [12] T_Loss: 0.7156 | V_Loss: 0.7143 | V_Acc: 0.8793 | LR: 0.000794
Epoch [13] T_Loss: 0.7082 | V_Loss: 0.6858 | V_Acc: 0.9138 | LR: 0.000761
Epoch [14] T_Loss: 0.7367 | V_Loss: 0